# 第7章：深入 PyTorch —— 模型构建与训练流程

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、模型构建** | §1-§2 | `nn.Sequential`、`nn.Module` 子类化 | Sequential, Module, forward() |
| **二、训练基础** | §3 | 数据加载、训练循环、验证循环 | DataLoader, train/eval, backward() |
| **三、扩展训练** | §4-§5 | torchmetrics、回调模式、模型保存 | torchmetrics, Callback, checkpoint |

> **学习建议**：建议从 §1 开始按顺序运行。
>
> 本笔记本使用 MNIST 真实数据集，所有代码可直接运行。

In [ ]:
# === 基础导入 ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

---
# 第一阶段：模型构建

本阶段学习 PyTorch 的两种建模方法，
掌握从简单到复杂的模型搭建方式：`nn.Sequential` 和 `nn.Module`。

---
## 7.1 PyTorch 工作流程

与 Keras 的渐进式复杂性不同，PyTorch 采用 **统一的建模哲学**：

| 特点 | 说明 |
|------|------|
| **统一 API** | 所有模型都继承自 `nn.Module`，没有多种构建方式 |
| **显式前向传播** | 数据流通过 `forward()` 方法定义，清晰透明 |
| **手动训练循环** | 没有 `fit()`，训练逻辑完全由你掌控 |

两种建模粒度：

| 方法 | 复杂度 | 适用场景 |
|------|--------|----------|
| **`nn.Sequential`** | 最易理解 | 层的简单堆叠，一条线结构 |
| **`nn.Module` 子类化** | 灵活 | 多输入/多输出/残差连接/任意控制流 |

---
## §1 `nn.Sequential` —— 简单模型的快速搭建

`nn.Sequential` 是 PyTorch 最简单的建模方式 —— 本质上是一个 **有序容器**，
所有层按顺序排列，数据从第一层流到最后一层。

**结构特征**：只能处理"一条线"的结构
```
Input → Layer1 → Layer2 → Layer3 → Output
```

**局限性**：不支持多输入、多输出、残差连接等复杂拓扑。

### 1.1 方式一：位置参数式初始化

最直观的写法 —— 直接按顺序传入层，一行代码搞定。

In [ ]:
# 位置参数式：按顺序直接传入层
model = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 10)
)

print(model)
print(f"\n注意：PyTorch 的层在创建时就有权重了（与 Keras 延迟构建不同）")
print(f"权重数量: {len(list(model.parameters()))}")

### 1.2 方式二：`OrderedDict` 命名式初始化

使用 `OrderedDict` 给每一层命名，方便调试和访问特定层。

In [ ]:
from collections import OrderedDict

# 命名式：用 OrderedDict 给每一层指定名称
model = nn.Sequential(OrderedDict([
    ("fc1", nn.Linear(784, 512)),
    ("relu1", nn.ReLU()),
    ("dropout1", nn.Dropout(0.5)),
    ("fc2", nn.Linear(512, 10))
]))

print(model)

# 通过名称访问特定层
print(f"\n可以通过名称访问层: model.fc1 = {model.fc1}")

### 1.3 方式三：使用 `add_module()` 动态添加

先创建空容器，再逐层添加。适合运行时动态决定层结构的场景。

In [ ]:
# 动态添加：先创建空 Sequential，再用 add_module() 逐层添加
model = nn.Sequential()
model.add_module("fc1", nn.Linear(784, 512))
model.add_module("relu1", nn.ReLU())
model.add_module("dropout1", nn.Dropout(0.5))
model.add_module("fc2", nn.Linear(512, 10))

print(model)
print(f"\n模型包含 {len(model)} 个模块")

### 1.4 PyTorch 与 Keras 的构建机制对比

| 特性 | Keras | PyTorch |
|------|-------|---------|
| 权重创建时机 | 延迟构建（收到输入或调用 build() 时） | 即时创建（层初始化时就有权重） |
| 输入形状推断 | 自动推断 | 需要在第一层显式指定 `in_features` |
| 查看结构 | `model.summary()` | `print(model)` 或 `torchinfo.summary()` |
| 访问参数 | `model.weights` | `model.parameters()` / `model.named_parameters()` |

In [ ]:
# 演示 PyTorch 的即时构建：创建后立刻有参数
model = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)

# 访问参数
print("[PyTorch] 创建后立即访问参数:")
for name, param in model.named_parameters():
    print(f"  {name}: shape={tuple(param.shape)}")

# 参数计算验证：
# Linear(784, 512): 784*512 + 512 = 401920 (权重 + 偏置)
# Linear(512, 10): 512*10 + 10 = 5130
# 总计: 401920 + 5130 = 407050
total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params}")

### 1.5 查看模型详细信息

使用 `torchinfo` 可以获得类似 Keras `model.summary()` 的详细报告。

In [ ]:
# 使用 torchinfo 查看详细模型摘要（类似 Keras 的 summary）
try:
    from torchinfo import summary
    model = nn.Sequential(
        nn.Linear(784, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10)
    )
    summary(model, input_size=(64, 784))  # 指定批量大小和输入形状
except ImportError:
    print("torchinfo 未安装，使用基本打印:")
    print(model)
    print(f"总参数: {sum(p.numel() for p in model.parameters())}")

### 1.6 动态修改 Sequential 模型

PyTorch 中可以通过索引或字典方式修改 Sequential 的层。

In [ ]:
# 动态修改 Sequential 模型
model = nn.Sequential(
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)

print("=== 修改之前 ===")
print(model)

# 方式一：通过索引替换层
model[2] = nn.Linear(512, 20)  # 将输出层从 10 改为 20

print("\n=== 替换最后一层之后 ===")
print(model)

# 方式二：添加新层（使用 add_module）
model.add_module("extra", nn.Linear(20, 5))

print("\n=== 添加新层之后 ===")
print(model)

### 第一阶段小结：`nn.Sequential`

| 概念 | 要点 |
|------|------|
| 位置参数式 | `Sequential(层1, 层2, ...)` 一行搞定 |
| `OrderedDict` | 给每一层命名，方便访问和调试 |
| `add_module()` | 适合运行时动态决定层结构 |
| 即时构建 | 权重在层创建时就有，不需要 build() |
| 修改层 | 通过索引 `model[i]` 或名称 `model.name` 替换 |
| 查看参数 | `model.parameters()` 或 `model.named_parameters()` |

---
## §2 `nn.Module` —— 灵活建模的核心方式

`nn.Module` 是 PyTorch 建模的 **基石**。所有层、所有模型都是 `nn.Module` 的子类。

与 Keras 的函数式 API 和模型子类化的区别：
- PyTorch **只有一种**建模方式：继承 `nn.Module` + 实现 `forward()`
- 既简单（理解容易）又灵活（控制力强）

```
简单 Sequential:   Input → L1 → L2 → Output

复杂 nn.Module:    ┌───── Input ─────┐
                   │                 │
                 Branch A         Branch B
                   │                 │
                   └──→ Merge ←─────┘
                          │
                       Output
```

### 2.1 `nn.Module` 核心概念

**两步**：
1. `__init__()` 中定义所有子层（"声明零件"）
2. `forward()` 中定义前向传播逻辑（"组装流程"）

In [ ]:
class SimpleClassifier(nn.Module):
    """
    简单的图像分类器。
    
    结构：Flatten → Linear(784→512) → ReLU → Dropout → Linear(512→10)
    
    与 Keras 函数式 API 的区别：
    - PyTorch 不区分"声明"和"连接"，所有逻辑都在 forward() 中
    - Keras 函数式 API: Layer(...)(input) 语法，Keras 推导计算图
    - PyTorch: 在 forward() 中直接调用层，更直观
    """
    
    def __init__(self):
        super().__init__()  # ⚠️ 必不可少！
        
        # 在 __init__ 中定义所有子层（声明零件）
        self.fc1 = nn.Linear(784, 512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        """
        定义前向传播逻辑。
        x: 输入张量, shape (batch_size, 784)
        返回: 输出 logits, shape (batch_size, 10)
        """
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# 实例化并测试
model = SimpleClassifier()
print(model)
print(f"\n总参数: {sum(p.numel() for p in model.parameters())}")

### 2.2 多输入模型

PyTorch 的 `forward()` 可以接收任意数量的参数，轻松实现多输入模型。

In [ ]:
class MultiInputModel(nn.Module):
    """
    多输入模型示例。
    
    场景：融合图像特征和元数据进行分类。
    - 输入1：图像特征 (batch, 784)
    - 输入2：元数据 (batch, 10)
    """
    
    def __init__(self):
        super().__init__()
        self.image_encoder = nn.Linear(784, 256)
        self.meta_encoder = nn.Linear(10, 32)
        self.classifier = nn.Linear(256 + 32, 10)

    def forward(self, image_features, meta_features):
        img_emb = torch.relu(self.image_encoder(image_features))
        meta_emb = torch.relu(self.meta_encoder(meta_features))
        
        # 拼接两个分支
        combined = torch.cat([img_emb, meta_emb], dim=1)
        return self.classifier(combined)

# 测试多输入
model = MultiInputModel()
dummy_img = torch.randn(32, 784)
dummy_meta = torch.randn(32, 10)
output = model(dummy_img, dummy_meta)
print(f"输出形状: {output.shape}")  # (32, 10)

### 2.3 多输出模型

返回 tuple 即可实现多输出。

In [ ]:
class MultiOutputModel(nn.Module):
    """
    多输出模型示例。
    
    场景：同时预测数字类别和数字是否为偶数（二分类）。
    """
    
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.digit_head = nn.Linear(512, 10)       # 10 类数字
        self.parity_head = nn.Linear(512, 2)       # 偶数/奇数

    def forward(self, x):
        features = self.features(x)
        digit_pred = self.digit_head(features)
        parity_pred = self.parity_head(features)
        return digit_pred, parity_pred  # 返回 tuple

# 测试多输出
model = MultiOutputModel()
dummy_input = torch.randn(32, 784)
digit_out, parity_out = model(dummy_input)
print(f"数字预测形状: {digit_out.shape}")    # (32, 10)
print(f"奇偶预测形状: {parity_out.shape}")   # (32, 2)

### 2.4 多输入多输出模型

结合前面的技巧，对应 Keras 函数式 API 的工单分类器示例。

In [ ]:
class TicketClassifier(nn.Module):
    """
    工单分类器 —— 对应 Keras 函数式 API 的多输入多输出示例。
    
    输入：
    - title: 标题特征 (batch, vocab_size)
    - text_body: 正文特征 (batch, vocab_size)
    - tags: 标签特征 (batch, num_tags)
    
    输出：
    - priority: 优先级 (batch, 1) sigmoid
    - department: 部门 (batch, num_departments) softmax
    """
    
    def __init__(self, vocab_size, num_tags, num_departments):
        super().__init__()
        
        # 计算拼接后的总维度
        total_input = vocab_size * 2 + num_tags
        
        self.shared = nn.Sequential(
            nn.Linear(total_input, 64),
            nn.ReLU(),
            nn.Dropout(0.5)
        )
        self.priority_head = nn.Linear(64, 1)
        self.department_head = nn.Linear(64, num_departments)

    def forward(self, title, text_body, tags):
        # 拼接三个输入
        x = torch.cat([title, text_body, tags], dim=1)
        features = self.shared(x)
        
        # 两个输出头
        priority = torch.sigmoid(self.priority_head(features))
        department = torch.softmax(self.department_head(features), dim=1)
        
        return priority, department

# 测试
vocab_size = 100
num_tags = 20
num_departments = 4
model = TicketClassifier(vocab_size, num_tags, num_departments)

dummy_title = torch.randn(32, vocab_size)
dummy_text = torch.randn(32, vocab_size)
dummy_tags = torch.randn(32, num_tags)
priority, department = model(dummy_title, dummy_text, dummy_tags)

print(f"priority 输出形状: {priority.shape}")      # (32, 1)
print(f"department 输出形状: {department.shape}")  # (32, 4)

### 2.5 使用 `nn.ModuleList` 和 `nn.ModuleDict` 管理动态层

当层的数量不固定时，用 `nn.ModuleList` 或 `nn.ModuleDict` 替代 Python 列表/字典。

In [ ]:
class DynamicModel(nn.Module):
    """
    使用 ModuleList 管理动态数量的层。
    
    ⚠️ 注意：不能用普通 Python list 存储 nn.Module，
    因为 list 中的 Module 不会被自动注册和追踪参数。
    """
    
    def __init__(self, num_hidden_layers=3, hidden_size=128):
        super().__init__()
        
        self.input_layer = nn.Linear(784, hidden_size)
        
        # 用 ModuleList 存储多个隐藏层
        self.hidden_layers = nn.ModuleList([
            nn.Linear(hidden_size, hidden_size) 
            for _ in range(num_hidden_layers)
        ])
        
        self.output_layer = nn.Linear(hidden_size, 10)
        
    def forward(self, x):
        x = torch.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = torch.relu(layer(x))
        return self.output_layer(x)

model = DynamicModel(num_hidden_layers=4, hidden_size=128)
dummy = torch.randn(32, 784)
output = model(dummy)
print(f"输出形状: {output.shape}")  # (32, 10)
print(f"总参数: {sum(p.numel() for p in model.parameters())}")

### 2.6 残差连接（ResNet 风格）

PyTorch 的核心优势：`forward()` 中可以用 **任意 Python 代码** 控制数据流。

In [ ]:
class ResidualBlock(nn.Module):
    """
    残差连接块。
    
    核心思想：output = F(x) + x
    - F(x) 是经过几层网络的处理
    - + x 是捷径连接（skip connection）
    """
    
    def __init__(self, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        identity = x  # 保存原始输入（捷径）
        
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        
        # 残差连接：处理结果 + 原始输入
        out = out + identity
        out = self.relu(out)
        return out

# 测试
block = ResidualBlock(128)
dummy = torch.randn(32, 128)
output = block(dummy)
print(f"残差块输出形状: {output.shape}")

### 2.7 从已有模型中提取特征创建新模型（特征复用）

对应 Keras 的"从中间层提取输出构建新模型"功能。
PyTorch 中通过访问模型的子模块来实现。

In [ ]:
# 1. 已有的基础模型
base_model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

# 2. 提取前几层作为特征提取器
# 方法一：切片 Sequential（适用于简单 Sequential 模型）
feature_extractor = nn.Sequential(*list(base_model.children())[:-1])  # 去掉最后一层

print("=== 特征提取器 ===")
print(feature_extractor)

# 测试
dummy = torch.randn(32, 784)
features = feature_extractor(dummy)
print(f"\n提取的特征形状: {features.shape}")  # (32, 128)

# 3. 在特征提取器基础上构建新模型
extended_model = nn.Sequential(
    feature_extractor,
    nn.Linear(128, 5)  # 新的输出层
)

output = extended_model(dummy)
print(f"扩展模型输出形状: {output.shape}")  # (32, 5)

print("\n注意：extended_model 复用了 base_model 前几层的权重！")

### 2.8 模型嵌套：将模型作为组件

一个 `nn.Module` 可以包含其他 `nn.Module` 作为子模块。

In [ ]:
class Encoder(nn.Module):
    """编码器组件"""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    """解码器组件"""
    def __init__(self, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

class AutoEncoder(nn.Module):
    """
    自编码器：组合 Encoder 和 Decoder。
    演示如何将已有模型嵌入到更大的模型中。
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim)
        self.decoder = Decoder(hidden_dim, input_dim)
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# 测试
ae = AutoEncoder(input_dim=784, hidden_dim=64)
dummy = torch.randn(32, 784)
reconstructed = ae(dummy)
print(f"重建输出形状: {reconstructed.shape}")  # (32, 784)
print(f"\n模型结构：")
print(ae)

### 第一阶段小结

| 概念 | 要点 |
|------|------|
| `nn.Sequential` | 简单模型，层按顺序堆叠，适合快速原型 |
| `nn.Module` | 灵活建模，`__init__` 定义层，`forward()` 定义数据流 |
| 多输入/多输出 | `forward()` 接收/返回多个张量即可 |
| `ModuleList`/`ModuleDict` | 管理动态数量的子层 |
| 残差连接 | `forward()` 中 `out = F(x) + x` |
| 特征复用 | 切片 `children()` 或提取子模块 |
| 模型嵌套 | `nn.Module` 可以包含其他 `nn.Module` |

---
# 第二阶段：训练流程

PyTorch 的训练流程与 Keras 完全不同 —— **没有 `compile()` 和 `fit()`**。

```
Keras:  compile → fit → evaluate → predict
         ↓
PyTorch: 定义优化器+损失 → 手写训练循环 → 验证 → 预测
```

但这恰恰是 PyTorch 的优势：**训练过程完全透明**，没有黑盒。

---
## §3 标准训练流程（MNIST 全流程）

### 3.1 数据加载与预处理

PyTorch 使用 `DataLoader` 管理数据批次，对应 Keras 的 `batch_size` 参数。

In [ ]:
# 3.1 加载 MNIST 数据

# 定义数据转换：转张量 + 归一化
transform = transforms.Compose([
    transforms.ToTensor(),                           # PIL → Tensor, [0,255] → [0,1]
    transforms.Normalize((0.1307,), (0.3081,))       # 标准化: (x - mean) / std
])

# 下载并加载数据集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

# 划分训练集和验证集
train_size = 50000
val_size = 10000
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

# 创建 DataLoader（数据加载器）
batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集: {len(train_subset)} 样本, {len(train_loader)} 个 batch")
print(f"验证集: {len(val_subset)} 样本, {len(val_loader)} 个 batch")
print(f"测试集:  {len(test_dataset)} 样本, {len(test_loader)} 个 batch")

# 查看一个 batch 的形状
for images, labels in train_loader:
    print(f"\n一个 batch: images={images.shape}, labels={labels.shape}")
    break

### 3.2 构建模型、优化器和损失函数

在 Keras 中，这三步合并在 `compile()` 中。
在 PyTorch 中，它们是独立的步骤，更加透明。

In [ ]:
# 3.2 构建模型
def get_mnist_model():
    """构建 MNIST 手写数字分类模型"""
    return nn.Sequential(
        nn.Flatten(),               # (B, 1, 28, 28) → (B, 784)
        nn.Linear(784, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10)
        # ⚠️ 注意：PyTorch 的 CrossEntropyLoss 内部已包含 Softmax
        # 所以模型输出不需要 Softmax，直接输出 logits 即可
    )

# 选择设备（GPU 优先）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建模型并移到目标设备
model = get_mnist_model().to(device)

# 定义优化器和损失函数（对应 Keras 的 compile）
optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

print(f"模型:\n{model}")
print(f"\n优化器: {optimizer}")
print(f"损失函数: {loss_fn}")

### 3.3 训练循环核心步骤

```
for epoch in range(epochs):
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()           # 1. 清零梯度
        outputs = model(batch_x)        # 2. 前向传播
        loss = loss_fn(outputs, y)      # 3. 计算损失
        loss.backward()                 # 4. 反向传播（计算梯度）
        optimizer.step()                # 5. 更新权重
```

**与 Keras 对比**：
```
Keras:  fit() 内部自动完成 1-5
PyTorch: 手动写 1-5，但每一步都清晰可见
```

In [ ]:
# 3.3 定义训练一个 epoch 的函数
def train_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    训练一个 epoch。
    
    核心步骤：
    1. model.train() → 设置训练模式（开启 Dropout/BN）
    2. optimizer.zero_grad() → 清零梯度
    3. 前向传播 → 计算损失
    4. loss.backward() → 反向传播计算梯度
    5. optimizer.step() → 更新权重
    """
    model.train()  # 设置为训练模式
    total_loss = 0
    correct = 0
    total = 0
    
    for step, (batch_x, batch_y) in enumerate(dataloader):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # 1. 清零梯度（不清零的话梯度会累加）
        optimizer.zero_grad()
        
        # 2. 前向传播
        outputs = model(batch_x)
        
        # 3. 计算损失
        loss = loss_fn(outputs, batch_y)
        
        # 4. 反向传播（自动计算所有参数的梯度）
        loss.backward()
        
        # 5. 更新权重
        optimizer.step()
        
        # 统计
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
        
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

### 3.4 验证循环

验证时不计算梯度，且需要关闭 Dropout。

In [ ]:
# 3.4 定义验证函数
@torch.no_grad()  # 装饰器：此函数内不计算梯度，节省内存和时间
def validate(model, dataloader, loss_fn, device):
    """
    验证模型性能。
    
    关键区别：
    - model.eval() → 设置评估模式（关闭 Dropout/BN 使用统计量）
    - torch.no_grad() → 不计算梯度，节省内存
    """
    model.eval()  # 设置为评估模式
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_x)
        loss = loss_fn(outputs, batch_y)
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

### 3.5 完整训练流程

In [ ]:
# 3.5 完整训练循环
epochs = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"开始训练，共 {epochs} 个 epoch\n")

for epoch in range(epochs):
    print(f"{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"{'='*50}")
    
    # 训练
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
    print(f"\n  训练损失: {train_loss:.4f}, 训练精度: {train_acc:.4f}")
    
    # 验证
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)
    print(f"  验证损失: {val_loss:.4f}, 验证精度: {val_acc:.4f}")
    
    # 记录历史
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

### 3.6 测试和预测（evaluate / predict）

对应 Keras 的 `evaluate()` 和 `predict()`。

In [ ]:
# 3.6 在独立测试集上评估（对应 Keras evaluate）
test_loss, test_acc = validate(model, test_loader, loss_fn, device)
print(f"测试集 - loss: {test_loss:.4f}, accuracy: {test_acc:.4f}")

In [ ]:
# 3.7 使用模型预测（对应 Keras predict）
@torch.no_grad()
def predict_samples(model, dataloader, device, num_samples=5):
    """预测前几个样本并展示结果"""
    model.eval()
    images, labels = next(iter(dataloader))
    images = images[:num_samples].to(device)
    labels = labels[:num_samples]
    
    outputs = model(images)
    probs = torch.softmax(outputs, dim=1)  # 转换为概率
    pred_classes = torch.argmax(probs, dim=1).cpu()
    
    print(f"{'样本':>6} | {'预测':>6} | {'概率':>8} | {'真实':>6} | 结果")
    print("-" * 48)
    for i in range(num_samples):
        match = "✓" if pred_classes[i] == labels[i] else "✗"
        print(f"  #{i:>4} | {pred_classes[i]:>6} | {probs[i, pred_classes[i]].item():>8.4f} | {labels[i]:>6} | {match}")

predict_samples(model, test_loader, device)

---
## §4 自定义指标（torchmetrics）

当内置指标不满足需求时，可以使用 `torchmetrics` 库。

**实现步骤**：继承 `torchmetrics.Metric` 类，实现三个方法：

| 方法 | 调用时机 | 职责 |
|------|---------|------|
| `__init__()` | 实例化时 | 定义状态变量（`self.add_state()`） |
| `update()` | 每个 batch 结束时 | 更新状态的计算逻辑 |
| `compute()` | 需要读取指标值时 | 返回当前的指标值 |
| `reset()` | 每个 epoch 开始时 | 重置状态变量 |

In [ ]:
# 4.1 使用 torchmetrics 内置指标
try:
    import torchmetrics
    print(f"torchmetrics 版本: {torchmetrics.__version__}")
    
    # 直接使用内置指标
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
    print("torchmetrics 可用，后续示例使用它")
except ImportError:
    print("torchmetrics 未安装，使用手动计算方式")
    print("安装方法: pip install torchmetrics")

In [ ]:
# 4.2 自定义 RMSE 指标

class RMSEMetric(torchmetrics.Metric):
    """
    自定义 RMSE（均方根误差）指标。
    
    数学公式：RMSE = sqrt( Σ(y_true - y_pred)² / N )
    """
    
    def __init__(self):
        super().__init__()
        # 定义状态变量（支持分布式自动归约）
        self.add_state("mse_sum", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("total", default=torch.tensor(0), dist_reduce_fx="sum")

    def update(self, preds, target):
        """每个 batch 调用：累加 MSE 和样本数"""
        # 将 preds 转为 one-hot 比较（或直接使用 logits 计算）
        probs = torch.softmax(preds, dim=1)
        one_hot = torch.nn.functional.one_hot(target, num_classes=probs.shape[1]).float()
        mse = torch.sum((one_hot - probs) ** 2)
        self.mse_sum += mse
        self.total += target.numel()

    def compute(self):
        """返回最终指标值"""
        return torch.sqrt(self.mse_sum / self.total)

# 测试自定义指标
rmse = RMSEMetric()
dummy_preds = torch.randn(32, 10)
dummy_target = torch.randint(0, 10, (32,))
rmse.update(dummy_preds, dummy_target)
print(f"RMSE: {rmse.compute():.4f}")

---
## §5 回调模式与模型保存

Keras 有内置的回调系统，PyTorch 中需要我们自己实现类似的模式。

### 回调基类定义

| 钩子 | 调用时机 |
|------|---------|
| `on_train_begin()` | 整个训练开始时 |
| `on_train_end()` | 整个训练结束时 |
| `on_epoch_begin(epoch)` | 每轮训练开始时 |
| `on_epoch_end(epoch, logs)` | 每轮训练结束时 |

### 常用回调

| 回调 | 作用 |
|------|------|
| `EarlyStopping` | 验证指标不再改善时提前停止 |
| `ModelCheckpoint` | 自动保存最佳模型权重 |
| `ReduceLROnPlateau` | 验证指标停滞时降低学习率 |

In [ ]:
# 5.1 回调基类

class Callback:
    """回调函数基类"""
    def on_train_begin(self, logs=None): pass
    def on_train_end(self, logs=None): pass
    def on_epoch_begin(self, epoch, logs=None): pass
    def on_epoch_end(self, epoch, logs=None): pass


class EarlyStopping(Callback):
    """
    早停回调。
    
    当监控指标在 patience 个 epoch 内没有改善时，停止训练。
    """
    def __init__(self, monitor="val_loss", patience=2, mode="min"):
        self.monitor = monitor
        self.patience = patience
        self.mode = mode  # "min" 或 "max"
        self.counter = 0
        self.best_value = None
        self.should_stop = False
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        # 判断是否改善
        if self.best_value is None:
            self.best_value = current
        elif (self.mode == "min" and current < self.best_value) or \
             (self.mode == "max" and current > self.best_value):
            self.best_value = current
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  [EarlyStopping] 指标 {self.monitor} 在 {self.patience} 轮内未改善，停止训练")


class ModelCheckpoint(Callback):
    """
    模型检查点回调。
    
    当监控指标改善时，保存模型权重。
    """
    def __init__(self, filepath, monitor="val_loss", mode="min"):
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.best_value = None
    
    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        
        if self.best_value is None or \
           (self.mode == "min" and current < self.best_value) or \
           (self.mode == "max" and current > self.best_value):
            self.best_value = current
            # 保存模型（推荐保存模型和优化器状态）
            torch.save({
                'epoch': epoch,
                'model_state_dict': logs.get('model').state_dict(),
                'optimizer_state_dict': logs.get('optimizer').state_dict(),
                'val_loss': current,
            }, self.filepath)
            print(f"  [Checkpoint] 模型已保存到 {self.filepath} (val_loss={current:.4f})")


print("回调类定义完成")

In [ ]:
# 5.2 组合使用回调的训练循环

def train_with_callbacks(model, optimizer, loss_fn, train_loader, val_loader, 
                         device, epochs, callbacks):
    """
    支持回调的训练循环。
    
    与 Keras 的 fit(callbacks=...) 类似，
    在训练的关键节点调用回调函数。
    """
    model = model.to(device)
    
    # on_train_begin
    for cb in callbacks:
        cb.on_train_begin()
    
    for epoch in range(epochs):
        # on_epoch_begin
        for cb in callbacks:
            cb.on_epoch_begin(epoch)
        
        # 训练
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
        
        # 验证
        val_loss, val_acc = validate(model, val_loader, loss_fn, device)
        
        logs = {
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "model": model, "optimizer": optimizer
        }
        
        print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
        
        # on_epoch_end
        for cb in callbacks:
            cb.on_epoch_end(epoch, logs)
        
        # 检查是否需要早停
        if any(cb.should_stop for cb in callbacks if hasattr(cb, 'should_stop')):
            break
    
    # on_train_end
    for cb in callbacks:
        cb.on_train_end()

# 使用示例
callbacks = [
    EarlyStopping(monitor="val_loss", patience=2, mode="min"),
    ModelCheckpoint(filepath="best_model.pt", monitor="val_loss", mode="min")
]

# 重新创建模型
model = get_mnist_model().to(device)
optimizer = optim.RMSprop(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

print("使用回调训练模型（最多 8 轮）：")
train_with_callbacks(model, optimizer, loss_fn, train_loader, val_loader,
                     device, epochs=8, callbacks=callbacks)

In [ ]:
# 5.3 加载保存的最优模型

import os
if os.path.exists("best_model.pt"):
    checkpoint = torch.load("best_model.pt", weights_only=False)
    
    # 恢复模型状态
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"[成功] 最优模型已加载 (保存于 epoch {checkpoint['epoch']+1}, val_loss={checkpoint['val_loss']:.4f})")
    
    # 在测试集上评估
    test_loss, test_acc = validate(model, test_loader, loss_fn, device)
    print(f"最优模型的测试精度: {test_acc:.4f}")
else:
    print("[提示] 未找到保存的模型文件")

### 第二阶段小结

| 步骤 | Keras | PyTorch |
|------|-------|---------|
| 配置训练 | `compile(optimizer, loss, metrics)` | 创建 `optimizer` + `loss_fn` |
| 训练 | `fit(x, y, epochs)` | 手写 `for` 循环 + `backward()` |
| 评估 | `evaluate(x, y)` | 手写验证循环 + `torch.no_grad()` |
| 预测 | `predict(x)` | `model(x)` + `torch.softmax()` |
| 扩展 | 自定义 Metric / Callback | `torchmetrics` / 自定义回调类 |
| 保存模型 | `model.save()` | `torch.save(model.state_dict(), ...)` |

---
# 总结

## PyTorch 核心工作流

```
1. 定义模型
   - nn.Sequential：简单的线性堆叠
   - nn.Module 子类化：灵活建模（推荐首选）

2. 准备训练组件
   - 优化器：optim.Adam / optim.RMSprop / optim.SGD
   - 损失函数：nn.CrossEntropyLoss / nn.MSELoss
   - 数据：DataLoader + Dataset

3. 训练循环
   - model.train()
   - optimizer.zero_grad()
   - outputs = model(batch_x)
   - loss = loss_fn(outputs, batch_y)
   - loss.backward()
   - optimizer.step()

4. 验证循环
   - model.eval() + torch.no_grad()

5. 扩展
   - 自定义 Metric → 继承 torchmetrics.Metric
   - 回调模式 → 自定义 Callback 类
   - 完全自定义训练 → 打开 7.4-自定义训练循环与高级用法.ipynb
```

## Keras vs PyTorch 建模对比

| 维度 | Keras | PyTorch |
|------|-------|---------|
| 简单模型 | `Sequential([层1, 层2])` | `nn.Sequential(层1, 层2)` |
| 复杂模型 | 函数式 API / 子类化 | `nn.Module` 子类化 |
| 前向传播 | 隐式（Keras 推导计算图） | 显式 `forward()` 方法 |
| 训练方式 | `compile()` + `fit()` | 手写训练循环 |
| 调试难度 | 较难（图模式） | 容易（即时执行） |

---
**下一步学习**：打开 `7.4-自定义训练循环与高级用法.ipynb` 学习自动微分详解、性能优化、封装训练器和 PyTorch Lightning。